# STEP 1 

In [0]:
df = spark.table("bronze.customer_complaints")
df.createOrReplaceTempView("bronze_raw")

In [0]:
total_bronze_rows = df.count()
print(f"Total rows read from Bronze layer: {total_bronze_rows}")

In [0]:
# when new complaints were added to the bronze layer  
display(df.groupBy("ingestion_timestamp").count())

In [0]:
# check for blank IDs (if any, the quarantine table will be 0)
from pyspark.sql import functions as F
display(df.filter(F.col("Complaint_ID").isNull()))

In [0]:
# isolating clean raws and dirty raws 
from pyspark.sql import functions as F 

raw_df = spark.table("bronze_raw")

quarantine_df = raw_df.filter((F.col("Complaint_ID").isNull()) | (F.trim(F.col("Complaint_ID")) == ""))
quarantine_rows_count = quarantine_df.count()

clean_df = raw_df.filter((F.col("Complaint_ID").isNotNull()) & (F.trim(F.col("Complaint_ID")) != ""))

In [0]:
# in case there are no ID nulls, the quarintine table will be empty so we handle it by only creating
# the quarntine table if any ID nulls exist  
if quarantine_rows_count > 0:
    quarantine_df.write.format("delta").mode("overwrite").saveAsTable("silver.complaints_quarantine")
    print(f"{quarantine_rows_count} bad rows written to quarantine table.")
else:
    print("No bad rows found in this batch. Quarantine table skip completed safely.")

In [0]:
# drop the empty column 
clean_df = clean_df.drop("Consumer_disputed")

# trim all the collumns 
string_columns = [item[0] for item in clean_df.dtypes if item[1] == "string"]
for col_name in string_columns :
    clean_df = clean_df.withColumn(col_name, F.trim(F.col(col_name)))


In [0]:
import re

# CONVERT every run of non-alphanumeric chars to a single underscore, then lowercase
def to_snake_case(name):
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()

for old_col in clean_df.columns:
    clean_df = clean_df.withColumnRenamed(old_col, to_snake_case(old_col))


In [0]:
clean_df.createOrReplaceTempView("silver_step1")

clean_rows_count = clean_df.count()

# this formula ensures that there are no missing rows 
assert total_bronze_rows == (clean_rows_count + quarantine_rows_count)

print(f"Step 1 Success!")
print(f"Total Bronze: {total_bronze_rows}")      
print(f"Silver Clean View: {clean_rows_count}")   
print(f"Quarantined Rows: {quarantine_rows_count}")

# STEP 2 

In [0]:
%sql
select * from silver_step1

In [0]:
%sql
-- Create or replace the temporary view for Step 2
create or replace temp view silver_step2 as 
select 

      cast(complaint_id as string) as complaint_id, -- convert it to string (unique identifier)
      to_date(date_received, 'M/d/yyyy') as date_received,
      to_date(date_sent_to_company, 'M/d/yyyy') as date_sent_to_company,
      
      -- Calculate the difference in days between the two dates
      datediff(to_date(date_sent_to_company, 'M/d/yyyy'), to_date(date_received, 'M/d/yyyy')) as days_to_send,
      case when upper(trim(timely_response)) = 'YES' then 1 
     when upper(trim(timely_response)) = 'NO' then 0 
     else null end as is_timely_response,
      product,
    sub_product,
    issue,
    sub_issue,
    consumer_complaint_narrative,
    company_public_response,
    company,
    state,
    zip_code,
    tags,
    consumer_consent_provided,
    submitted_via,
    company_response_to_consumer,
    ingestion_timestamp,
    source_system

from silver_step1      

In [0]:
df_step1 = spark.table("silver_step1")
df_step2 = spark.table("silver_step2")

assert df_step2.count() == df_step1.count(), f"Row count mismatch! Step 1 has {df_step1.count()} rows, but Step 2 has {df_step2.count()} rows."

print("Step 2 Completed Successfully with ZERO row loss")
print(f"Verified Row Count: {df_step2.count()} rows.")

# **STEP 3**

In [0]:
%sql
-- temporary view for Step3
create or replace temp view silver_step3 as
select

      complaint_id,
      date_received,
      date_sent_to_company,
      days_to_send,
      is_timely_response,
      product,
      sub_product,
      issue,

      -- fill null sub_issue 
      case when sub_issue is null or trim(sub_issue) = '' then 'Not Applicable'
           else sub_issue
      end as sub_issue,

      
      consumer_complaint_narrative, --  narrative untouched

      -- fill null company_public_response 
      case when company_public_response is null or trim(company_public_response) = '' then 'No Public Response'
           else company_public_response
      end as company_public_response,

      company,

      -- fill null state with 'Unknown', standardize outlier to 'UM'
      case when state is null or trim(state) = '' then 'Unknown'
           when upper(trim(state)) = 'UNITED STATES MINOR OUTLYING ISLANDS' then 'UM'
           else state
      end as state,

      zip_code,

      -- fill null tags
      case when tags is null or trim(tags) = '' then 'None'
           else tags
      end as tags,

      -- fill null consumer_consent_provided 
      case when consumer_consent_provided is null or trim(consumer_consent_provided) = '' then 'Unknown'
           else consumer_consent_provided
      end as consumer_consent_provided,

      -- fill null submitted_via 
      case when submitted_via is null or trim(submitted_via) = '' then 'Unknown'
           else submitted_via
      end as submitted_via,

      company_response_to_consumer,
      ingestion_timestamp,
      source_system

from silver_step2

In [0]:
df_step2 = spark.table("silver_step2")
df_step3 = spark.table("silver_step3")

assert df_step3.count() == df_step2.count(), f"Row count mismatch! Step 2 has {df_step2.count()} rows, but Step 3 has {df_step3.count()} rows."

print("Step 3 Completed Successfully with ZERO row loss")
print(f"Verified Row Count: {df_step3.count()} rows.")

# **STEP 4**

In [0]:
%sql
-- Step 4: enrichment — add derived columns (row count unchanged)
create or replace temp view silver_step4 as
select
      complaint_id,
      date_received,
      date_sent_to_company,
      days_to_send,
      is_timely_response,
      product,
      sub_product,
      issue,
      sub_issue,

      -- narrative kept raw, plus two derived columns (read BEFORE any fill)
      consumer_complaint_narrative,
      case when consumer_complaint_narrative is null
                or trim(consumer_complaint_narrative) = '' then 0
           else 1
      end as has_narrative,
      case when consumer_complaint_narrative is null
                or trim(consumer_complaint_narrative) = '' then null
           else length(consumer_complaint_narrative)
      end as narrative_length,

      company_public_response,
      company,
      company_response_to_consumer,

      -- case-insensitive match (same lesson as the timely_response fix)
      case when lower(trim(company_response_to_consumer)) = 'in progress' then 1
           else 0
      end as is_in_progress,

      state,
      zip_code,
      -- first 3 chars of zip; fully-masked (XXXXX) or blank -> Unknown
      case when zip_code is null or trim(zip_code) = '' then 'Unknown'
           when upper(substring(zip_code, 1, 3)) = 'XXX' then 'Unknown'
           else substring(zip_code, 1, 3)
      end as zip3,

      tags,
      consumer_consent_provided,
      submitted_via,
      ingestion_timestamp,
      source_system
from silver_step3

In [0]:
df_step3 = spark.table("silver_step3")
df_step4 = spark.table("silver_step4")

# enrichment must NOT change the row count
assert df_step4.count() == df_step3.count(), \
    f"Row count changed in Step 4! step3={df_step3.count()}, step4={df_step4.count()}"

# persist to the Silver Delta table
df_step4.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.customer_complaints")

print("Step 4 complete —silver.customer complaints written.")
print(f"Rows: {df_step4.count()}  |  Columns: {len(df_step4.columns)}")

In [0]:
bronze_count     = spark.table("bronze.customer_complaints").count()
silver_count     = spark.table("silver.customer_complaints").count()
quarantine_count = spark.table("silver.complaints_quarantine").count()

assert bronze_count == silver_count + quarantine_count, \
    f"Reconciliation FAILED: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}"

print("Reconciliation passed:")
print(f"  bronze ({bronze_count}) = silver ({silver_count}) + quarantine ({quarantine_count})")

In [0]:
%sql
select * from silver.customer_complaints limit(10)